# 02. 추론 파라미터 실험

AI가 텍스트를 생성할 때 사용하는 **파라미터**를 직접 바꿔가며 결과를 비교합니다.  
같은 질문에 파라미터만 달리해도 응답이 크게 달라지는 것을 체험해보세요.

---

## 토큰(Token)이란?

LLM은 글자 단위가 아닌 **토큰** 단위로 텍스트를 처리합니다.

```
예시 문장: "안녕하세요, 반갑습니다!"

토큰화 결과 (모델마다 다름):
["안녕", "하세요", ",", " 반", "갑습니다", "!"]
  토큰1   토큰2   토큰3  토큰4    토큰5     토큰6

→ 6 토큰

영어: "Hello world" = ["Hello", " world"] = 2 토큰
한국어는 영어보다 토큰 수가 많습니다 (1글자 ≈ 1~2 토큰)
```

---

## 주요 파라미터 요약

| 파라미터 | 역할 | 범위 | 기본값 |
|----------|------|------|--------|
| `temperature` | 창의성/무작위성 조절 | 0.0 ~ 2.0 | 1.0 |
| `top_p` | 확률 상위 p% 단어만 선택 | 0.0 ~ 1.0 | 1.0 |
| `max_tokens` | 최대 생성 토큰 수 | 1 ~ 모델한계 | 모델마다 다름 |
| `stop` | 이 단어 나오면 생성 중단 | 문자열 목록 | None |

In [ ]:
# 공통 설정 — vLLM이 Session 터미널에서 실행 중이어야 합니다 (포트 8001)
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8001/v1",
    api_key="dummy",
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

SYSTEM_PROMPT = "너는 10년 경력의 청담동 패션 스타일리스트야. 친근한 말투로 답변해줘."

def ask(question, **kwargs):
    """vLLM에 질문을 보내고 응답과 토큰 수를 출력하는 헬퍼 함수"""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        **kwargs
    )
    content = response.choices[0].message.content
    tokens = response.usage.completion_tokens
    finish = response.choices[0].finish_reason
    return content, tokens, finish

print("✅ 설정 완료. 아래 셀들을 순서대로 실행하세요.")

## 실험 1. temperature — 창의성 조절

```
temperature = 0.0  → 항상 가장 확률 높은 단어 선택 (결정론적, 딱딱함)
temperature = 0.7  → 적당한 무작위성 (일반적으로 권장)
temperature = 1.5  → 매우 창의적이지만 가끔 이상한 답변
```

같은 질문을 3번 보내서 얼마나 다른 답변이 나오는지 비교해봅니다.

In [ ]:
question = "오늘 카페 데이트에 어울리는 여성 캐주얼 코디 추천해줘"

for temp in [0.0, 0.7, 1.5]:
    content, tokens, finish = ask(question, temperature=temp, max_tokens=150)
    print(f"\n{'='*55}")
    print(f"🌡️  temperature = {temp}")
    print(f"{'='*55}")
    print(content)
    print(f"\n→ 생성 토큰: {tokens}개, 종료 이유: {finish}")

## 실험 2. max_tokens — 응답 길이 제한

```
max_tokens = 30   → 짧게 잘림 (finish_reason = 'length')
max_tokens = 100  → 적당한 길이
max_tokens = 300  → 길게 답변 가능 (finish_reason = 'stop')
```

`finish_reason`이 `'stop'`이면 자연스럽게 완성된 것,  
`'length'`이면 토큰 한도에 잘린 것입니다.

In [ ]:
question = "봄 시즌 패션 트렌드를 자세히 설명해줘"

for max_tok in [30, 100, 300]:
    content, tokens, finish = ask(question, temperature=0.7, max_tokens=max_tok)
    print(f"\n{'='*55}")
    print(f"📏 max_tokens = {max_tok}")
    print(f"{'='*55}")
    print(content)
    print(f"\n→ 실제 생성: {tokens}토큰, 종료 이유: '{finish}'")
    if finish == "length":
        print("  ⚠️  토큰 한도에 잘렸습니다 — max_tokens를 늘리면 전체 답변을 볼 수 있어요")

## 실험 3. top_p — 단어 선택 범위 조절

```
top_p = 0.1  → 확률 상위 10% 단어만 선택 → 안전하고 예측 가능한 답변
top_p = 0.9  → 확률 상위 90% 단어에서 선택 → 다양하고 자연스러운 답변
top_p = 1.0  → 모든 단어 후보 (제한 없음)
```

`temperature`와 `top_p`는 함께 작용합니다.  
일반적으로 `temperature=0.7, top_p=0.9` 조합을 많이 사용합니다.

In [ ]:
question = "지성 피부에 맞는 스킨케어 루틴 추천해줘"

for top_p_val in [0.1, 0.9]:
    content, tokens, finish = ask(
        question,
        temperature=0.7,
        top_p=top_p_val,
        max_tokens=150
    )
    print(f"\n{'='*55}")
    print(f"🎯 top_p = {top_p_val}")
    print(f"{'='*55}")
    print(content)

## 실험 4. 패션 챗봇 최적 파라미터 찾기

실험들을 통해 배운 내용을 바탕으로, 패션/뷰티 챗봇에 가장 적합한 파라미터를 설정해봅니다.

**패션/뷰티 추천 챗봇의 특성:**
- 창의적이면서도 실용적인 추천이 필요
- 너무 엉뚱하거나 이상한 답변은 안 됨
- 적당한 길이 (너무 짧으면 내용이 부족, 너무 길면 읽기 싫음)

In [ ]:
# 패션 챗봇 최적 파라미터 조합
OPTIMAL_PARAMS = {
    "temperature": 0.7,   # 창의적이지만 안정적
    "top_p": 0.9,         # 다양한 표현 허용
    "max_tokens": 300,    # 충분한 추천 내용
}

test_questions = [
    "내일 격식 있는 자리에 입고 갈 만한 30대 남성 비즈니스 캐주얼 추천해줘",
    "지성 피부에 맞는 수분크림 조합 알려줘",
]

for q in test_questions:
    content, tokens, finish = ask(q, **OPTIMAL_PARAMS)
    print(f"\n{'='*55}")
    print(f"❓ 질문: {q}")
    print(f"{'='*55}")
    print(content)
    print(f"\n→ 토큰: {tokens}개")

## 정리

| 파라미터 | 낮은 값 | 높은 값 | 패션 챗봇 권장값 |
|----------|---------|---------|------------------|
| `temperature` | 딱딱하고 반복적 | 창의적이지만 불안정 | **0.7** |
| `top_p` | 안전하지만 단조로움 | 다양하고 자연스러움 | **0.9** |
| `max_tokens` | 답변이 잘림 | 너무 길어짐 | **200~300** |

---

다음 노트북: **`03_cloudera_intro.ipynb`** — Cloudera CML 배포 이해